# Amplitude amplification

This notebook prepares a one-qubit state with initial good-state probability $1/4$, marks $|1\rangle$ as good, and applies one amplitude-amplification iteration. The final success probability is exactly one.

The convention follows Brassard, Høyer, Mosca, and Tapp, [*Quantum Amplitude Amplification and Estimation*](https://arxiv.org/abs/quant-ph/0005055), Eqs. (1), (5), and (8).

In [ ]:
import numpy as np

import qarp
from qarp.algorithms import AmplitudeAmplification, Sampler
from qarp.blocks import PhaseShiftBlock, SimpleBlock

## Define the state preparation and oracle

With $\theta=\pi/6$, applying $R_y(2\theta)$ to $|0\rangle$ produces $\cos(\theta)|0\rangle+\sin(\theta)|1\rangle$. Thus the initial probability of the good state $|1\rangle$ is $\sin^2(\theta)=1/4$. `PhaseShiftBlock(pi)` implements $\operatorname{diag}(1,-1)$ and therefore marks $|1\rangle$.

In [ ]:
theta = np.pi / 6

state_preparation = SimpleBlock(1, name="A")
state_preparation.ry(0, 2 * theta)

oracle = PhaseShiftBlock(np.pi, name="O_good")

## Run one amplification iteration

For one iteration, the expected probability is $\sin^2((2\cdot1+1)\theta)=\sin^2(\pi/2)=1$. `good_states=[1]` is classical reporting metadata used to calculate `success_probability`; the oracle itself defines the quantum marking operation.

In [ ]:
algorithm = AmplitudeAmplification(
    state_preparation,
    oracle,
    n_iterations=1,
    good_states=[1],
    primitive=Sampler(n_shots=qarp.EXACT),
).build()

distribution = algorithm.run()

assert np.isclose(distribution.get((1,), 0.0), 1.0, atol=1e-12)
assert np.isclose(algorithm.success_probability, 1.0, atol=1e-12)

print("Distribution:", distribution)
print("Success probability:", algorithm.success_probability)